## Phase 4: Random Forest

Testing whether a different model family can raise the AUPRC ceiling
found in Phase 3 (~0.74), or whether that's a limit of the data itself
rather than logistic regression specifically.

Random Forest is an ensemble of decision trees, each trained on a
random subset of rows and features, averaged together. Unlike logistic
regression, it doesn't need feature scaling, trees split each feature
independently on its own threshold, so different ranges (Amount vs
the V columns) don't affect training the way they can for gradient
descent based models.

Trained on the original, untouched X_train/y_train (no SMOTE, no
class_weight yet), same split as every prior model, so this is a
direct comparison against the plain baseline first.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
)

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())  # sanity check: should both be ~0.0017, matching every prior phase

(227845, 30) (56962, 30)
0.001729245759178389 0.0017204452090867595


In [2]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
y_scores_rf = model_rf.predict_proba(X_test)[:, 1]

cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_rf)

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
auprc_rf = average_precision_score(y_test, y_scores_rf)

print(f"\nPrecision: {precision_rf:.4f}  (baseline 0.8312)")
print(f"Recall:    {recall_rf:.4f}  (baseline 0.6531)")
print(f"AUPRC:     {auprc_rf:.4f}  (baseline 0.7427)")

Confusion matrix [[TN, FP], [FN, TP]]:
[[56859     5]
 [   18    80]]

Precision: 0.9412  (baseline 0.8312)
Recall:    0.8163  (baseline 0.6531)
AUPRC:     0.8734  (baseline 0.7427)


## Phase 4 result: Random Forest

Confusion matrix: TP=80, FN=18, FP=5, TN=56,859.

| Metric    | Baseline (LR) | Random Forest |
|-----------|---------------|----------------|
| Precision | 0.8312        | 0.9412         |
| Recall    | 0.6531        | 0.8163         |
| AUPRC     | 0.7427        | 0.8734         |

Random Forest improves precision, recall, and AUPRC simultaneously,
something no logistic regression variant achieved in Phase 3 (every
attempt there traded one for the other, and AUPRC stayed capped
around 0.72 to 0.74 regardless of technique).

This revises Phase 3's tentative conclusion: the AUPRC ceiling wasn't
a property of the dataset, it was specific to logistic regression's
linear decision boundary. Random Forest's ability to model non-linear
feature interactions appears to genuinely unlock a better fraud
detector on this data, not just a recalibrated one.

No feature scaling or resampling was needed to get this result, it
used the original, untouched, imbalanced training data as-is.